# L01 · Causalidad elemental

**S1 · Lectura: Cap. 1 · 40–55 min**

## La pregunta

¿La diferencia que vemos es un efecto o solo una asociación?

## Qué harás

Separar asociación, efecto y sesgo de selección.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una lectura corta de tres ejemplos y de la paradoja de Simpson.

> **Idea clave:** Un resultado observado no dice qué habría pasado con la misma persona bajo la otra opción.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = setup()

In [ ]:
# Una imagen para empezar: asociación y causalidad
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.axis("off")
for x, label in zip((0.18, 0.5, 0.82), ("Causa común", "Tratamiento", "Resultado")):
    ax.text(x, 0.55, label, ha="center", va="center", fontsize=12, bbox={"boxstyle": "round,pad=.5", "alpha": .15})
ax.annotate("puede explicar", (0.36, 0.55), (0.25, 0.55), arrowprops={"arrowstyle": "->"}, ha="center")
ax.annotate("pregunta causal", (0.68, 0.55), (0.58, 0.55), arrowprops={"arrowstyle": "->"}, ha="center")
ax.set_title("Antes de estimar: ¿qué relación estamos suponiendo?")
plt.tight_layout()
plt.show()

## 1. Asociativa o causal (§1.1)
> «Los estudiantes con beca rinden más.» ¿La beca **causa** el rendimiento o las becas se dan a quienes ya rendían?

La primera frase describe una **asociación** (lo que vemos). La pregunta «¿qué pasaría si se la quitamos?» es **causal** (lo que *haríamos*). Hay tres formas típicas de confundirlas; las simulamos para que veas el mecanismo, no solo el nombre.

## 2. Tres escenas (§1.2)
### Escena A — las cigüeñas
Ciudades más grandes tienen más cigüeñas **y** más nacimientos. Una tercera variable (el tamaño) produce la asociación.

In [ ]:
n = 400
tamano = rng.lognormal(0, .6, n)                    # confusor: tamaño de la ciudad
cigue = 5 * tamano + rng.normal(0, 1, n)
naci = 30 * tamano + rng.normal(0, 6, n)
def partial_corr(a, b, z):
    ra = a - np.polyval(np.polyfit(z, a, 1), z); rb = b - np.polyval(np.polyfit(z, b, 1), z)
    return np.corrcoef(ra, rb)[0, 1]
print(f"corr(cigüeñas, nacimientos)           = {np.corrcoef(cigue, naci)[0,1]:+.2f}")
print(f"corr parcial dado el tamaño           = {partial_corr(cigue, naci, tamano):+.2f}")

**Mira el resultado.** ¿Por qué desaparece la asociación al controlar el tamaño?

**Respuesta:**

Porque las cigüeñas no entran en la ecuación de los nacimientos: ambas variables **comparten causa** (el tamaño). Al comparar ciudades *del mismo tamaño* la relación se esfuma.



### Escena B — pie y lectura
El tamaño del pie «predice» la comprensión lectora. La edad es la causa común.

In [ ]:
edad = rng.integers(5, 13, 600)
pie = 14 + 1.1 * edad + rng.normal(0, 1, 600)
lectura = 20 + 6 * edad + rng.normal(0, 8, 600)
print(f"corr(pie, lectura) global        = {np.corrcoef(pie, lectura)[0,1]:+.2f}")
dentro = [np.corrcoef(pie[edad == e], lectura[edad == e])[0, 1] for e in range(5, 13)]
print("corr dentro de cada edad (media) =", f"{np.mean(dentro):+.2f}")

### Escena C — Simpson en un hospital
Dos tratamientos, **A** y **B**. La gravedad del paciente decide quién recibe cuál: los casos graves van mayormente a B.

In [ ]:
def hospital(rng):
    filas = []
    for trat, grave, leve, p_g, p_l in [("A", 200, 800, .60, .93), ("B", 800, 200, .65, .95)]:
        for sev, k, p in [("grave", grave, p_g), ("leve", leve, p_l)]:
            filas += [(trat, sev, rng.binomial(1, p)) for _ in range(k)]
    return pd.DataFrame(filas, columns=["trat", "gravedad", "sobrevive"])
hosp = hospital(rng)
hosp.head()

In [ ]:
# ✏️ Completa tú — tasas_por_grupo
# Pista: df.groupby([...])['sobrevive'].mean()
def tasas_por_grupo(df, por):
    """Tasa de supervivencia por tratamiento, agrupando además por `por` (lista de columnas, puede ser [])."""
    raise NotImplementedError("agrupa por ['trat'] + por y promedia 'sobrevive'")

In [ ]:
print("GLOBAL:\n", tasas_por_grupo(hosp, []).to_string())
print("\nPOR GRAVEDAD:\n", tasas_por_grupo(hosp, ["gravedad"]).to_string())
lab_check("tasas_por_grupo", lambda d, p: tasas_por_grupo(d, p).to_numpy(),
          [((hosp, []), hosp.groupby("trat")["sobrevive"].mean().round(3).to_numpy(), 1e-9)])

**Mira el resultado.** ¿Qué tratamiento «parece mejor» en el agregado y cuál en cada estrato? ¿Con cuál te quedas para decidir?

**Respuesta:**

A parece mejor en el agregado y **B es mejor en cada estrato**. Para decidir qué dar a un paciente de una gravedad dada, vale la comparación **dentro** del estrato (la gravedad es confusor: causa de la asignación y del resultado). El agregado mezcla *quién* recibe cada tratamiento con *qué hace* el tratamiento.



## 3. Resultados potenciales y sesgo de selección (§1.3)
Cada persona tiene dos resultados posibles, $Y(1)$ y $Y(0)$, pero **solo vemos uno** (problema fundamental). Aquí construimos una población donde, **por una vez**, vemos ambos para poder pesar el sesgo.

Con $\Delta = E[Y|T=1]-E[Y|T=0]$ (lo que calcula cualquiera):

$$\Delta \;=\; \underbrace{E[Y(1)-Y(0)\mid T=1]}_{\text{ATT}} \;+\; \underbrace{E[Y(0)\mid T=1]-E[Y(0)\mid T=0]}_{\text{sesgo de selección}}\qquad(1.1)$$

In [ ]:
N = 200_000
motivacion = rng.normal(size=N)
y0 = 50 + 8 * motivacion + rng.normal(0, 5, N)       # rendimiento sin el programa
y1 = y0 + 4                                          # efecto verdadero = 4 para todos
# Selección: los MENOS motivados piden el programa (por ejemplo, un refuerzo)
t = (rng.uniform(size=N) < 1 / (1 + np.exp(1.2 * motivacion))).astype(int)
y = np.where(t == 1, y1, y0)                          # lo único que se observa
print(f"Δ observado         = {y[t==1].mean() - y[t==0].mean():+.3f}")
print(f"efecto verdadero    = {np.mean(y1 - y0):+.3f}")

In [ ]:
# ✏️ Completa tú — sesgo_seleccion
# Pista: Solo se puede calcular porque aquí conocemos y0 de todos.
def sesgo_seleccion(y0, t):
    """E[Y(0)|T=1] - E[Y(0)|T=0]"""
    raise NotImplementedError("promedia y0 entre tratados y entre controles y réstalos")

In [ ]:
att = np.mean((y1 - y0)[t == 1]); sesgo = sesgo_seleccion(y0, t)
delta = y[t == 1].mean() - y[t == 0].mean()
print(f"ATT = {att:+.3f} · sesgo de selección = {sesgo:+.3f} · ATT + sesgo = {att + sesgo:+.3f} · Δ = {delta:+.3f}")
assert abs(delta - (att + sesgo)) < 1e-6, "la identidad (1.1) debe cumplirse exactamente"
lab_check("sesgo_seleccion", sesgo_seleccion, [((y0, t), y0[t==1].mean() - y0[t==0].mean(), 1e-9)])

**Mira el resultado.** El programa ayuda (+4) pero Δ sale negativo. ¿Qué dice el libro que ocurre?

**Respuesta:**

Los que se inscriben ya rendían menos (su $Y(0)$ es menor): el **sesgo de selección** es negativo y grande y se come el efecto. Ningún modelo predictivo corrige eso sin un supuesto sobre cómo se asignó el tratamiento.



**La aleatorización anula el sesgo.** Si repartimos el programa con una moneda, $T$ ya no depende de $Y(0)$:

In [ ]:
t_rand = rng.binomial(1, .5, N)
y_r = np.where(t_rand == 1, y1, y0)
print(f"sesgo de selección con moneda = {sesgo_seleccion(y0, t_rand):+.3f}")
print(f"Δ con moneda                  = {y_r[t_rand==1].mean() - y_r[t_rand==0].mean():+.3f}   (efecto verdadero = 4)")

## 4. La escalera (§1.4) y por qué el ML no es causal por defecto (§1.5)
| Nivel | Pregunta | Ejemplo |
|---|---|---|
| 1 · Asociación | ¿Qué *veo*? | ¿Quienes tienen beca rinden más? |
| 2 · Intervención | ¿Qué pasa si *hago*? | ¿Qué pasa si **otorgo** becas a todos? |
| 3 · Contrafactual | ¿Qué habría pasado si…? | ¿Habría rendido María igual sin beca? |

Un modelo predictivo excelente responde al nivel 1. Lo comprobamos con la población anterior: ajustamos $Y$ sobre $T$ (sin la motivación).

In [ ]:
from sklearn.linear_model import LinearRegression
m = LinearRegression().fit(t.reshape(-1, 1), y)
print(f"R² del modelo predictivo  = {m.score(t.reshape(-1,1), y):.3f}")
print(f"'efecto' que lee del modelo (coef de T) = {m.coef_[0]:+.3f}   | efecto causal verdadero = +4.000")

**Mira el resultado.** Aunque el R² fuera alto, ¿serviría el coeficiente de T para decidir si dar el programa?

**Respuesta:**

No: **predecir bien no identifica el efecto**. El coeficiente recoge la asociación (incluido el sesgo de selección). Para el efecto hace falta un supuesto de identificación (aleatorizar, ajustar por la motivación si se midiera, etc.). Esa es la tesis del curso.



## 5. Supuesto y bitácora
El supuesto que sostiene toda lectura causal de este lab es que **conocemos la historia de la asignación**. En la vida real eso es *no observable*: se declara.

In [ ]:
fila = bitacora(
    pregunta="¿El programa mejora el rendimiento o los inscritos ya rendían distinto?",
    metodo="Descomposición Δ = ATT + sesgo con resultados potenciales simulados",
    estimando="Δ observado, ATT y sesgo de selección",
    numero_ic=f"Δ = {delta:+.2f}; ATT = {att:+.2f}; sesgo = {sesgo:+.2f} (simulación, sin IC)",
    supuesto=supuesto("no observable", "declarado y acotado", "la asignación depende de la motivación, que en la práctica no se mide"),
    decision="revisar identificación", lab="L01")

---
## Qué queda para el libro
Ejercicios del libro cubiertos: clasificar niveles y construir una asociación espuria propia con confusor. Quedan para casa: *por qué ningún algoritmo da el efecto individual* y el caso del *R² = 0,95*.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*